In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
import pandas as pd
import math
from datetime import datetime, timedelta
from difflib import SequenceMatcher
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import seaborn as sns
import numpy as np
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
from lifelines import CoxTimeVaryingFitter
from lifelines import CoxPHFitter
import warnings
from Preproces_prod2 import *
from patsy import dmatrices
warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [3]:
path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

In [4]:
df = pd.read_csv(path_data/'df_281024_s31_all_meses_all_group.csv')
df_upc = pd.read_csv(path_data/'df_UPC_31_2810_all_meses_.csv')

In [5]:
df.NOMBRE_REGION.nunique(),df_upc.NOMBRE_REGION.nunique()

(16, 16)

In [9]:
chile_chico = ['METROPOLITANA','VALPARAISO','LOS LAGOS','LOS RIOS','MAULE','TARAPACA'] #,"O'HIGGINS"
df_chile_chico = df.copy().query('NOMBRE_REGION.isin(@chile_chico)')
df_upc_chile_chico = df_upc.copy().query('NOMBRE_REGION.isin(@chile_chico)')

In [17]:
df_print = df_chile_chico.copy()

print("N_bebes_total:", df_print.drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", df_print.drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
      ", i.e cobertura = ",
      round(df_print.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
            /df_print.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_VRS:", df_print.drop_duplicates(subset='RUN', keep='last').event.sum())

print('\n')

print("N_bebes_muy_prematuros_total:", df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_muy_prematuros_inmune:", df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
      ", i.e cobertura = ",
      round(df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
            /df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_muy_prematuros_VRS:", df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').event.sum())


N_bebes_total: 99483
N_bebes_inmune: 92033.0 , i.e cobertura =  92.51 %
N_bebes_VRS: 518


N_bebes_muy_prematuros_total: 8841
N_bebes_muy_prematuros_inmune: 8236.0 , i.e cobertura =  93.16 %
N_bebes_muy_prematuros_VRS: 73


In [18]:
df_print = df_upc.copy().query('NOMBRE_REGION.isin(@chile_chico)')
print('------------------UPC------------------')
print("N_bebes_total:", df_print.drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", df_print.drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
      ", i.e cobertura = ",
      round(df_print.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
            /df_print.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_UPC:", df_print.drop_duplicates(subset='RUN', keep='last').event.sum())

print('\n')

print("N_bebes_muy_prematuros_total:", df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_muy_prematuros_inmune:", df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
      ", i.e cobertura = ",
      round(df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
            /df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_muy_prematuros_UPC:", df_print.query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').event.sum())

------------------UPC------------------
N_bebes_total: 99619
N_bebes_inmune: 92313.0 , i.e cobertura =  92.67 %
N_bebes_UPC: 122


N_bebes_muy_prematuros_total: 8864
N_bebes_muy_prematuros_inmune: 8274.0 , i.e cobertura =  93.34 %
N_bebes_muy_prematuros_UPC: 25


In [14]:
ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(df_chile_chico.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"')[['start', 'inmunizado', 'stop', 'RUN', 'event','SEXO','NOMBRE_REGION','group_age','muy_prematuro']], # one_hot_cols_educ[1:] + #, 'Macrozona2', 'muy_prematuro','INS_N_M'
          id_col="RUN", event_col="event", start_col="start", stop_col="stop", strata=['NOMBRE_REGION','group_age'])
print("N_bebes_total:", df_chile_chico.drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", df_chile_chico.drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
              ", i.e cobertura = ",
              round(df_chile_chico.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                    /df_chile_chico.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_VRS:", df_chile_chico.drop_duplicates(subset='RUN', keep='last').event.sum())
display(printSummary(ctv_0))

N_bebes_total: 86206
N_bebes_inmune: 79840.0 , i.e cobertura =  92.62 %
N_bebes_VRS: 478


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-1.542,0.786,0.109,-1.757,-1.328,0.735,0.827,0.0
SEXO,0.560,-0.751,0.096,0.373,0.747,-1.112,-0.451,0.0
muy_prematuro,1.081,-1.947,0.241,0.609,1.553,-3.724,-0.839,0.0


In [ ]:
#.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"')

In [11]:
ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(df_upc_chile_chico[['start', 'inmunizado', 'stop', 'RUN', 'event','SEXO','NOMBRE_REGION','group_age','muy_prematuro']], # one_hot_cols_educ[1:] + #, 'Macrozona2', 'muy_prematuro','INS_N_M'
          id_col="RUN", event_col="event", start_col="start", stop_col="stop", strata=['NOMBRE_REGION','group_age'])
print("N_bebes_total:", df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
              ", i.e cobertura = ",
              round(df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                    /df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_VRS:", df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').event.sum())
display(printSummary(ctv_0))

N_bebes_total: 86331
N_bebes_inmune: 80093.0 , i.e cobertura =  92.77 %
N_bebes_VRS: 116


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-1.682,0.814,0.216,-2.106,-1.259,0.716,0.878,0.000
SEXO,0.576,-0.780,0.194,0.195,0.957,-1.605,-0.216,0.003
muy_prematuro,0.955,-1.599,0.510,-0.044,1.954,-6.058,0.043,0.061


In [24]:
ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(df_chile_chico.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"').query('32<=SEMANAS<=36')[['start', 'inmunizado', 'stop', 'RUN', 'event','SEXO','NOMBRE_REGION','group_age','muy_prematuro']], # one_hot_cols_educ[1:] + #, 'Macrozona2', 'muy_prematuro','INS_N_M'
          id_col="RUN", event_col="event", start_col="start", stop_col="stop", strata=['NOMBRE_REGION','group_age'])
print("N_bebes_total:", df_chile_chico.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"').query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", df_chile_chico.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"').query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
              ", i.e cobertura = ",
              round(df_chile_chico.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"').query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                    /df_chile_chico.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"').query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_VRS:", df_chile_chico.query('RUN!="bed99009d64eb031ead9235037fc95761d6f334e1d0bc27be4349f1734ca5b2f"').query('32<=SEMANAS<=36').drop_duplicates(subset='RUN', keep='last').event.sum())
display(printSummary(ctv_0))

N_bebes_total: 7673
N_bebes_inmune: 7158.0 , i.e cobertura =  93.29 %
N_bebes_VRS: 70


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-0.879,0.585,0.329,-1.525,-0.234,0.208,0.782,0.008
SEXO,0.541,-0.717,0.258,0.034,1.047,-1.848,-0.035,0.036
muy_prematuro,0.229,-0.257,0.466,-0.684,1.141,-2.131,0.495,0.623


In [16]:
ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(df_upc_chile_chico.query('32<=SEMANAS<=36')[['start', 'inmunizado', 'stop', 'RUN', 'event','SEXO','NOMBRE_REGION','group_age','muy_prematuro']], # one_hot_cols_educ[1:] + #, 'Macrozona2', 'muy_prematuro','INS_N_M'
          id_col="RUN", event_col="event", start_col="start", stop_col="stop", strata=['NOMBRE_REGION','group_age'])
print("N_bebes_total:", df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').inmunizado.sum(),
              ", i.e cobertura = ",
              round(df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                    /df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_VRS:", df_upc_chile_chico.drop_duplicates(subset='RUN', keep='last').event.sum())
display(printSummary(ctv_0))

N_bebes_total: 86331
N_bebes_inmune: 80093.0 , i.e cobertura =  92.77 %
N_bebes_VRS: 116


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-1.560,0.790,0.494,-2.528,-0.592,0.447,0.920,0.002
SEXO,0.437,-0.548,0.435,-0.415,1.290,-2.631,0.340,0.315
muy_prematuro,-0.430,0.349,1.031,-2.450,1.591,-3.909,0.914,0.677
